# Teacher Models

This notebook trains and evaluates sequence-based teacher models
for the two-stage bearing fault diagnosis system.

Stage 1:
Healthy vs Damaged
Feature set: All22

Stage 2:
Outer vs Inner vs Both
Feature set: Top12

Models:
- LSTM
- GRU
- Transformer

The final test bearings remain frozen and are not used for model tuning.

In [ ]:
from pathlib import Path
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

BASE_DIR = str(Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd())
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

print("Environment ready.")

In [ ]:
stage1_data = joblib.load(
    os.path.join(PROCESSED_DIR, "stage1_scaled_sequences.joblib")
)

stage2_data = joblib.load(
    os.path.join(PROCESSED_DIR, "stage2_scaled_sequences.joblib")
)

print("Stage 1 keys:", stage1_data.keys())
print("Stage 2 keys:", stage2_data.keys())

In [ ]:
for feature_set in ["top_12", "top_15", "all_22"]:
    X_train, y_train = stage1_data[feature_set]["X_train"], stage1_data[feature_set]["y_train"]
    X_val, y_val = stage1_data[feature_set]["X_val"], stage1_data[feature_set]["y_val"]
    X_test, y_test = stage1_data[feature_set]["X_test"], stage1_data[feature_set]["y_test"]

    print(f"\nStage 1 - {feature_set}")
    print("Train:", X_train.shape, y_train.shape)
    print("Val:  ", X_val.shape, y_val.shape)
    print("Test: ", X_test.shape, y_test.shape)

In [ ]:
# Stage 1 LSTM Teacher - All22 features

X_train = stage1_data["all_22"]["X_train"]
y_train = stage1_data["all_22"]["y_train"]

X_val = stage1_data["all_22"]["X_val"]
y_val = stage1_data["all_22"]["y_val"]

X_test = stage1_data["all_22"]["X_test"]
y_test = stage1_data["all_22"]["y_test"]

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)
print("Classes:", np.unique(y_train))

In [ ]:
# Stage 1 LSTM Teacher

lstm_model = keras.Sequential([
    layers.Input(shape=(5, 22)),
    layers.LSTM(64),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

lstm_model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Handle the Stage 1 class imbalance
classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights)
}

print("Class weights:", class_weight_dict)

# Early stopping prevents unnecessary training
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

# Train using training data and validation data only
lstm_history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining complete.")
print("Epochs actually run:", len(lstm_history.history["loss"]))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Validation predictions
val_prob = lstm_model.predict(X_val, verbose=0).ravel()
val_pred = (val_prob >= 0.5).astype(int)

print("Stage 1 LSTM - Validation Results")
print("----------------------------------")
print("Accuracy:          ", accuracy_score(y_val, val_pred))
print("Balanced Accuracy: ", balanced_accuracy_score(y_val, val_pred))
print("Precision:         ", precision_score(y_val, val_pred))
print("Recall:            ", recall_score(y_val, val_pred))
print("F1 Score:          ", f1_score(y_val, val_pred))
print("ROC-AUC:           ", roc_auc_score(y_val, val_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=["Healthy", "Damaged"]
    )
)

In [ ]:
# Stage 1 GRU Teacher - All22 features

gru_model = keras.Sequential([
    layers.Input(shape=(5, 22)),
    layers.GRU(64),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

gru_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

gru_model.summary()

In [ ]:
# Train Stage 1 GRU

gru_history = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining complete.")
print("Epochs actually run:", len(gru_history.history["loss"]))

In [ ]:
# Stage 1 GRU - Validation Evaluation

val_prob_gru = gru_model.predict(X_val, verbose=0).ravel()
val_pred_gru = (val_prob_gru >= 0.5).astype(int)

print("Stage 1 GRU - Validation Results")
print("--------------------------------")
print("Accuracy:          ", accuracy_score(y_val, val_pred_gru))
print("Balanced Accuracy: ", balanced_accuracy_score(y_val, val_pred_gru))
print("Precision:         ", precision_score(y_val, val_pred_gru))
print("Recall:            ", recall_score(y_val, val_pred_gru))
print("F1 Score:          ", f1_score(y_val, val_pred_gru))
print("ROC-AUC:           ", roc_auc_score(y_val, val_prob_gru))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_gru))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred_gru,
        target_names=["Healthy", "Damaged"]
    )
)

In [ ]:
# Check the physical bearings represented in the validation sequences

features_df = pd.read_csv(
    os.path.join(PROCESSED_DIR, "paderborn_vibration_features.csv")
)

val_bearings = set(
    features_df[
        features_df["bearing"].isin(
            ["K005", "KA03", "KA08", "KI03", "KB24"]
        )
    ]["bearing"].unique()
)

print("Validation bearings:", sorted(val_bearings))

In [ ]:
# Measurement-level class distribution for validation bearings

val_df = features_df[
    features_df["bearing"].isin(
        ["K005", "KA03", "KA08", "KI03", "KB24"]
    )
].copy()

print(
    val_df.groupby(["bearing", "binary_label"])
    .size()
    .unstack(fill_value=0)
)

print("\nValidation bearing/fault mapping:")
print(
    val_df[["bearing", "fault_location"]]
    .drop_duplicates()
    .sort_values("bearing")
)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Build an explicit feature-extraction graph using the trained GRU layers
embedding_input = keras.Input(shape=(5, 22))

x = embedding_input

# Use every layer except the final sigmoid output layer
for layer in gru_model.layers[:-1]:
    x = layer(x, training=False)

embedding_model = keras.Model(
    inputs=embedding_input,
    outputs=x
)

# Extract the learned 32-dimensional representation
val_embeddings = embedding_model.predict(X_val, verbose=0)

print("GRU embedding shape:", val_embeddings.shape)

# PCA -> 2 dimensions
pca_gru = PCA(n_components=2)
val_pca = pca_gru.fit_transform(val_embeddings)

print("Explained variance ratio:", pca_gru.explained_variance_ratio_)
print("Total explained variance:", pca_gru.explained_variance_ratio_.sum())

# Visualization
plt.figure(figsize=(9, 7))

healthy = y_val == 0
damaged = y_val == 1

plt.scatter(
    val_pca[healthy, 0],
    val_pca[healthy, 1],
    label="Healthy",
    alpha=0.7
)

plt.scatter(
    val_pca[damaged, 0],
    val_pca[damaged, 1],
    label="Damaged",
    alpha=0.7
)

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("GRU Learned Representation - Stage 1 Validation Data")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# Function to create a completely fresh Stage 1 Transformer

def build_transformer():
    transformer_input = keras.Input(shape=(5, 22))

    x = layers.Dense(32)(transformer_input)

    attention_output = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.1
    )(x, x)

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(64, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(32)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    output = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(
        inputs=transformer_input,
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall")
        ]
    )

    return model

In [ ]:
# Batch-size experiment
# Every batch size gets a completely fresh Transformer

batch_sizes = [32, 64]

transformer_batch_results = []

for batch_size in batch_sizes:

    print("\n" + "=" * 50)
    print(f"Training fresh Transformer | Batch size = {batch_size}")
    print("=" * 50)

    # Start with a clean Keras session
    tf.keras.backend.clear_session()
    # Recreate a completely new modeltf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(10)

    transformer_model = build_transformer()

    # Fresh callback for this experiment
    early_stopping_exp = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = transformer_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=[early_stopping_exp],
        verbose=1
    )

    # Validation prediction
    val_prob = transformer_model.predict(
        X_val,
        verbose=0
    ).ravel()

    val_pred = (val_prob >= 0.5).astype(int)

    result = {
        "batch_size": batch_size,
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_balanced_accuracy": balanced_accuracy_score(y_val, val_pred),
        "val_precision": precision_score(y_val, val_pred),
        "val_recall": recall_score(y_val, val_pred),
        "val_f1": f1_score(y_val, val_pred),
        "val_roc_auc": roc_auc_score(y_val, val_prob)
    }

    transformer_batch_results.append(result)

    print("\nValidation result:")
    print(result)

In [ ]:
transformer_model.save(
    os.path.join(results_dir, "stage1_transformer_final_batch32.keras")
)

In [ ]:
# Final Stage 1 LSTM Teacher

def build_lstm():
    model = keras.Sequential([
        layers.Input(shape=(5, 22)),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall")
        ]
    )

    return model

In [ ]:
# Train final Stage 1 LSTM
# Fixed seed = 10, batch size = 32

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(10)

lstm_model_final = build_lstm()

early_stopping_lstm = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

lstm_history_final = lstm_model_final.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping_lstm],
    verbose=1
)

print("\nFinal LSTM training complete.")
print("Epochs actually run:", len(lstm_history_final.history["loss"]))
print("Parameters:", lstm_model_final.count_params())

In [ ]:
# Save final Stage 1 LSTM teacher

results_dir = os.path.join(BASE_DIR, "results")
os.makedirs(results_dir, exist_ok=True)

lstm_model_final.save(
    os.path.join(
        results_dir,
        "stage1_lstm_teacher_final.keras"
    )
)

print("Saved:")
print(
    os.path.join(
        results_dir,
        "stage1_lstm_teacher_final.keras"
    )
)

In [ ]:
# Verify saved LSTM teacher

lstm_path = os.path.join(
    results_dir,
    "stage1_lstm_teacher_final.keras"
)

print("File exists:", os.path.exists(lstm_path))
print("File path:", lstm_path)

# Validation prediction
val_prob_lstm_final = lstm_model_final.predict(
    X_val,
    verbose=0
).ravel()

val_pred_lstm_final = (
    val_prob_lstm_final >= 0.5
).astype(int)

print("\nStage 1 Final LSTM - Validation Results")
print("---------------------------------------")
print("Accuracy:          ", accuracy_score(y_val, val_pred_lstm_final))
print("Balanced Accuracy: ", balanced_accuracy_score(y_val, val_pred_lstm_final))
print("Precision:         ", precision_score(y_val, val_pred_lstm_final))
print("Recall:            ", recall_score(y_val, val_pred_lstm_final))
print("F1 Score:          ", f1_score(y_val, val_pred_lstm_final))
print("ROC-AUC:           ", roc_auc_score(y_val, val_prob_lstm_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_lstm_final))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred_lstm_final,
        target_names=["Healthy", "Damaged"]
    )
)

In [ ]:
# Check whether the original LSTM teacher is still available

print("Original lstm_model exists:", "lstm_model" in globals())

if "lstm_model" in globals():
    print("Parameters:", lstm_model.count_params())

    try:
        test_prediction = lstm_model.predict(X_val[:2], verbose=0)
        print("Original LSTM is still usable.")
        print("Sample predictions:", test_prediction.ravel())
    except Exception as e:
        print("Original LSTM exists but cannot currently be used.")
        print("Error:", e)

In [ ]:
# Save the original Stage 1 LSTM teacher

original_lstm_path = os.path.join(
    results_dir,
    "stage1_lstm_teacher_original.keras"
)

lstm_model.save(original_lstm_path)

print("Original LSTM teacher saved:")
print(original_lstm_path)

In [ ]:
# Recalculate original LSTM validation metrics

original_lstm_prob = lstm_model.predict(
    X_val,
    verbose=0
).ravel()

original_lstm_pred = (
    original_lstm_prob >= 0.5
).astype(int)

original_lstm_results = {
    "model": "LSTM",
    "feature_set": "All22",
    "batch_size": 32,
    "parameters": lstm_model.count_params(),
    "val_accuracy": accuracy_score(y_val, original_lstm_pred),
    "val_balanced_accuracy": balanced_accuracy_score(y_val, original_lstm_pred),
    "val_precision": precision_score(y_val, original_lstm_pred),
    "val_recall": recall_score(y_val, original_lstm_pred),
    "val_f1": f1_score(y_val, original_lstm_pred),
    "val_roc_auc": roc_auc_score(y_val, original_lstm_prob)
}

print(original_lstm_results)

In [ ]:
# Check whether the original GRU teacher is still available

print("Original gru_model exists:", "gru_model" in globals())

if "gru_model" in globals():
    print("Parameters:", gru_model.count_params())

    try:
        test_prediction = gru_model.predict(X_val[:2], verbose=0)
        print("Original GRU is still usable.")
        print("Sample predictions:", test_prediction.ravel())
    except Exception as e:
        print("Original GRU exists but cannot currently be used.")
        print("Error:", e)

In [ ]:
# Save the original Stage 1 GRU teacher

original_gru_path = os.path.join(
    results_dir,
    "stage1_gru_teacher_original.keras"
)

gru_model.save(original_gru_path)

print("Original GRU teacher saved:")
print(original_gru_path)

In [ ]:
# Recalculate original GRU validation metrics

original_gru_prob = gru_model.predict(
    X_val,
    verbose=0
).ravel()

original_gru_pred = (
    original_gru_prob >= 0.5
).astype(int)

original_gru_results = {
    "model": "GRU",
    "feature_set": "All22",
    "batch_size": 32,
    "parameters": gru_model.count_params(),
    "val_accuracy": accuracy_score(y_val, original_gru_pred),
    "val_balanced_accuracy": balanced_accuracy_score(y_val, original_gru_pred),
    "val_precision": precision_score(y_val, original_gru_pred),
    "val_recall": recall_score(y_val, original_gru_pred),
    "val_f1": f1_score(y_val, original_gru_pred),
    "val_roc_auc": roc_auc_score(y_val, original_gru_prob)
}

print(original_gru_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, original_gru_pred))

In [ ]:
# Final Stage 1 Transformer Teacher
# Fixed configuration: All22, batch size 32, seed 10

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(10)

transformer_model_final = build_transformer()

transformer_model_final.summary()

In [ ]:
# Train final Stage 1 Transformer

early_stopping_transformer = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

transformer_history_final = transformer_model_final.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping_transformer],
    verbose=1
)

print("\nFinal Transformer training complete.")
print("Epochs actually run:",
      len(transformer_history_final.history["loss"]))
print("Parameters:",
      transformer_model_final.count_params())

In [ ]:
# Save final Stage 1 Transformer teacher

transformer_final_path = os.path.join(
    results_dir,
    "stage1_transformer_teacher_final.keras"
)

transformer_model_final.save(transformer_final_path)

print("Saved:")
print(transformer_final_path)

In [ ]:
# Stage 1 Final Transformer - Validation Evaluation

val_prob_transformer_final = transformer_model_final.predict(
    X_val,
    verbose=0
).ravel()

val_pred_transformer_final = (
    val_prob_transformer_final >= 0.5
).astype(int)

print("Stage 1 Final Transformer - Validation Results")
print("----------------------------------------------")
print("Accuracy:          ",
      accuracy_score(y_val, val_pred_transformer_final))
print("Balanced Accuracy: ",
      balanced_accuracy_score(y_val, val_pred_transformer_final))
print("Precision:         ",
      precision_score(y_val, val_pred_transformer_final))
print("Recall:            ",
      recall_score(y_val, val_pred_transformer_final))
print("F1 Score:          ",
      f1_score(y_val, val_pred_transformer_final))
print("ROC-AUC:           ",
      roc_auc_score(y_val, val_prob_transformer_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_transformer_final))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred_transformer_final,
        target_names=["Healthy", "Damaged"]
    )
)

print("\nSaved model exists:",
      os.path.exists(transformer_final_path))

In [ ]:
# ============================================================
# Final Stage 1 Teacher Test Evaluation
# Test set is used only now, after all configurations are fixed.
# ============================================================

lstm_saved = keras.models.load_model(
    os.path.join(results_dir, "stage1_lstm_teacher_original.keras")
)

gru_saved = keras.models.load_model(
    os.path.join(results_dir, "stage1_gru_teacher_original.keras")
)

transformer_saved = keras.models.load_model(
    os.path.join(results_dir, "stage1_transformer_teacher_final.keras")
)

teacher_models = {
    "LSTM": lstm_saved,
    "GRU": gru_saved,
    "Transformer": transformer_saved
}

stage1_test_results = []

for model_name, model in teacher_models.items():

    test_prob = model.predict(X_test, verbose=0).ravel()
    test_pred = (test_prob >= 0.5).astype(int)

    result = {
        "model": model_name,
        "parameters": model.count_params(),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred),
        "test_recall": recall_score(y_test, test_pred),
        "test_f1": f1_score(y_test, test_pred),
        "test_roc_auc": roc_auc_score(y_test, test_prob)
    }

    stage1_test_results.append(result)

    print("\n" + "=" * 55)
    print(f"{model_name} - Stage 1 Test Results")
    print("=" * 55)

    print("Accuracy:          ", result["test_accuracy"])
    print("Balanced Accuracy: ", result["test_balanced_accuracy"])
    print("Precision:         ", result["test_precision"])
    print("Recall:            ", result["test_recall"])
    print("F1 Score:          ", result["test_f1"])
    print("ROC-AUC:           ", result["test_roc_auc"])

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, test_pred))

print("\n\nFinal Stage 1 Teacher Comparison")
display(pd.DataFrame(stage1_test_results))

In [ ]:
# Save final Stage 1 teacher test results

stage1_test_df = pd.DataFrame(stage1_test_results)

stage1_test_path = os.path.join(
    results_dir,
    "stage1_teacher_test_results.csv"
)

stage1_test_df.to_csv(
    stage1_test_path,
    index=False
)

display(stage1_test_df)

print("Saved:", stage1_test_path)

In [ ]:
# Per-bearing Stage 1 test accuracy

test_bearings = ["K006", "KA22", "KB27", "KI18", "KI21"]

# Number of test sequences belonging to each bearing
# The sequence generator preserves one label per sequence.
bearing_sequence_counts = {
    "K006": 64,
    "KA22": 64,
    "KB27": 64,
    "KI18": 64,
    "KI21": 64
}

per_bearing_results = []

for model_name, model in teacher_models.items():

    test_prob = model.predict(X_test, verbose=0).ravel()
    test_pred = (test_prob >= 0.5).astype(int)

    # The current frozen Stage 1 test sequences are ordered
    # by bearing according to the sequence-generation process.
    start = 0

    for bearing, count in bearing_sequence_counts.items():

        end = start + count

        bearing_acc = accuracy_score(
            y_test[start:end],
            test_pred[start:end]
        )

        per_bearing_results.append({
            "model": model_name,
            "bearing": bearing,
            "accuracy": bearing_acc
        })

        start = end

per_bearing_df = pd.DataFrame(per_bearing_results)

display(
    per_bearing_df.pivot(
        index="bearing",
        columns="model",
        values="accuracy"
    )
)

In [ ]:
per_bearing_path = os.path.join(
    results_dir,
    "stage1_teacher_per_bearing_test_results.csv"
)

per_bearing_df.to_csv(
    per_bearing_path,
    index=False
)

print("Saved:", per_bearing_path)

In [ ]:
# ============================================================
# Stage 2 Teacher Models
# Outer vs Inner vs Both
# Feature set = Top12
# ============================================================

X_train_s2 = stage2_data["top_12"]["X_train"]
y_train_s2 = stage2_data["top_12"]["y_train"]

X_val_s2 = stage2_data["top_12"]["X_val"]
y_val_s2 = stage2_data["top_12"]["y_val"]

X_test_s2 = stage2_data["top_12"]["X_test"]
y_test_s2 = stage2_data["top_12"]["y_test"]

print("Stage 2 data:")
print("Train:", X_train_s2.shape, y_train_s2.shape)
print("Val:  ", X_val_s2.shape, y_val_s2.shape)
print("Test: ", X_test_s2.shape, y_test_s2.shape)

print("\nTraining class distribution:")
print(
    pd.Series(y_train_s2)
    .value_counts()
    .sort_index()
)

print("\nValidation class distribution:")
print(
    pd.Series(y_val_s2)
    .value_counts()
    .sort_index()
)

print("\nTest class distribution:")
print(
    pd.Series(y_test_s2)
    .value_counts()
    .sort_index()
)

In [ ]:
# Stage 2 class weights
# Classes:
# 0 = Outer
# 1 = Inner
# 2 = Both

classes_s2 = np.unique(y_train_s2)

class_weights_s2 = compute_class_weight(
    class_weight="balanced",
    classes=classes_s2,
    y=y_train_s2
)

class_weight_dict_s2 = {
    int(cls): float(weight)
    for cls, weight in zip(classes_s2, class_weights_s2)
}

print("Stage 2 class weights:")
for cls, weight in class_weight_dict_s2.items():
    label = {
        0: "Outer",
        1: "Inner",
        2: "Both"
    }[cls]

    print(f"{cls} ({label}): {weight:.4f}")

In [ ]:
# Stage 2 LSTM Teacher - Top12 features
# Classes: 0 = Outer, 1 = Inner, 2 = Both

def build_lstm_stage2():
    model = keras.Sequential([
        layers.Input(shape=(5, 12)),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


lstm_stage2 = build_lstm_stage2()

lstm_stage2.summary()

In [ ]:
# Train Stage 2 LSTM Teacher

early_stopping_s2_lstm = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

lstm_stage2_history = lstm_stage2.fit(
    X_train_s2,
    y_train_s2,
    validation_data=(X_val_s2, y_val_s2),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict_s2,
    callbacks=[early_stopping_s2_lstm],
    verbose=1
)

print("\nStage 2 LSTM training complete.")
print(
    "Epochs actually run:",
    len(lstm_stage2_history.history["loss"])
)

In [ ]:
# Stage 2 LSTM - Validation Evaluation

val_prob_s2_lstm = lstm_stage2.predict(
    X_val_s2,
    verbose=0
)

val_pred_s2_lstm = np.argmax(
    val_prob_s2_lstm,
    axis=1
)

print("Stage 2 LSTM - Validation Results")
print("---------------------------------")
print(
    "Accuracy:          ",
    accuracy_score(y_val_s2, val_pred_s2_lstm)
)
print(
    "Balanced Accuracy: ",
    balanced_accuracy_score(y_val_s2, val_pred_s2_lstm)
)
print(
    "Macro Precision:   ",
    precision_score(
        y_val_s2,
        val_pred_s2_lstm,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro Recall:      ",
    recall_score(
        y_val_s2,
        val_pred_s2_lstm,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro F1:          ",
    f1_score(
        y_val_s2,
        val_pred_s2_lstm,
        average="macro",
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_s2,
        val_pred_s2_lstm
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_val_s2,
        val_pred_s2_lstm,
        target_names=["Outer", "Inner", "Both"],
        zero_division=0
    )
)

In [ ]:
# Save the final Stage 2 LSTM teacher

stage2_lstm_path = os.path.join(
    results_dir,
    "stage2_lstm_teacher_final.keras"
)

lstm_stage2.save(stage2_lstm_path)

print("Stage 2 LSTM teacher saved:")
print(stage2_lstm_path)

In [ ]:
# Stage 2 Transformer Teacher - Top12 features
# Classes: 0 = Outer, 1 = Inner, 2 = Both

def build_transformer_stage2():
    transformer_input = keras.Input(shape=(5, 12))

    # Project 12 input features to 32 dimensions
    x = layers.Dense(32)(transformer_input)

    # Self-attention
    attention_output = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.1
    )(x, x)

    # Residual connection + normalization
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    # Feed-forward network
    ffn = layers.Dense(64, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(32)(ffn)

    # Residual connection + normalization
    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    # Convert sequence to one vector
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    # 3-class output
    output = layers.Dense(
        3,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs=transformer_input,
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


transformer_stage2 = build_transformer_stage2()

transformer_stage2.summary()

In [ ]:
# Train Stage 2 Transformer Teacher

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(10)

transformer_stage2 = build_transformer_stage2()

early_stopping_s2_transformer = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

transformer_stage2_history = transformer_stage2.fit(
    X_train_s2,
    y_train_s2,
    validation_data=(X_val_s2, y_val_s2),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict_s2,
    callbacks=[early_stopping_s2_transformer],
    verbose=1
)

print("\nStage 2 Transformer training complete.")
print(
    "Epochs actually run:",
    len(transformer_stage2_history.history["loss"])
)
print(
    "Parameters:",
    transformer_stage2.count_params()
)

In [ ]:
# Save final Stage 2 Transformer teacher

stage2_transformer_path = os.path.join(
    results_dir,
    "stage2_transformer_teacher_final.keras"
)

transformer_stage2.save(stage2_transformer_path)

print("Stage 2 Transformer teacher saved:")
print(stage2_transformer_path)
print("Parameters:", transformer_stage2.count_params())

In [ ]:
# Stage 2 Transformer - Validation Evaluation

val_prob_s2_transformer = transformer_stage2.predict(
    X_val_s2,
    verbose=0
)

val_pred_s2_transformer = np.argmax(
    val_prob_s2_transformer,
    axis=1
)

print("Stage 2 Transformer - Validation Results")
print("-----------------------------------------")
print(
    "Accuracy:          ",
    accuracy_score(y_val_s2, val_pred_s2_transformer)
)
print(
    "Balanced Accuracy: ",
    balanced_accuracy_score(
        y_val_s2,
        val_pred_s2_transformer
    )
)
print(
    "Macro Precision:   ",
    precision_score(
        y_val_s2,
        val_pred_s2_transformer,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro Recall:      ",
    recall_score(
        y_val_s2,
        val_pred_s2_transformer,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro F1:          ",
    f1_score(
        y_val_s2,
        val_pred_s2_transformer,
        average="macro",
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_s2,
        val_pred_s2_transformer
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_val_s2,
        val_pred_s2_transformer,
        target_names=["Outer", "Inner", "Both"],
        zero_division=0
    )
)

In [ ]:
# ============================================================
# Stage 2 Teacher Validation Comparison
# ============================================================

stage2_teacher_validation_results = [
    {
        "model": "LSTM",
        "parameters": lstm_stage2.count_params(),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred_s2_lstm
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred_s2_lstm
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred_s2_lstm,
            average="macro",
            zero_division=0
        )
    },
    {
        "model": "Transformer",
        "parameters": transformer_stage2.count_params(),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred_s2_transformer
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred_s2_transformer
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred_s2_transformer,
            average="macro",
            zero_division=0
        )
    }
]

stage2_teacher_validation_df = pd.DataFrame(
    stage2_teacher_validation_results
)

display(stage2_teacher_validation_df)

stage2_teacher_validation_df.to_csv(
    os.path.join(
        results_dir,
        "stage2_teacher_validation_results.csv"
    ),
    index=False
)

print("Stage 2 teacher validation results saved.")

In [ ]:
# ============================================================
# Final Stage 2 Teacher Test Evaluation
# ============================================================

stage2_teacher_models = {
    "LSTM": keras.models.load_model(
        os.path.join(
            results_dir,
            "stage2_lstm_teacher_final.keras"
        )
    ),
    "Transformer": keras.models.load_model(
        os.path.join(
            results_dir,
            "stage2_transformer_teacher_final.keras"
        )
    )
}

stage2_test_results = []

for model_name, model in stage2_teacher_models.items():

    test_prob = model.predict(
        X_test_s2,
        verbose=0
    )

    test_pred = np.argmax(
        test_prob,
        axis=1
    )

    result = {
        "model": model_name,
        "parameters": model.count_params(),
        "test_accuracy": accuracy_score(
            y_test_s2,
            test_pred
        ),
        "test_balanced_accuracy": balanced_accuracy_score(
            y_test_s2,
            test_pred
        ),
        "test_macro_precision": precision_score(
            y_test_s2,
            test_pred,
            average="macro",
            zero_division=0
        ),
        "test_macro_recall": recall_score(
            y_test_s2,
            test_pred,
            average="macro",
            zero_division=0
        ),
        "test_macro_f1": f1_score(
            y_test_s2,
            test_pred,
            average="macro",
            zero_division=0
        )
    }

    stage2_test_results.append(result)

    print("\n" + "=" * 55)
    print(f"{model_name} - Stage 2 Test Results")
    print("=" * 55)

    print("Accuracy:          ", result["test_accuracy"])
    print("Balanced Accuracy: ", result["test_balanced_accuracy"])
    print("Macro Precision:   ", result["test_macro_precision"])
    print("Macro Recall:      ", result["test_macro_recall"])
    print("Macro F1:          ", result["test_macro_f1"])

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_test_s2,
            test_pred
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test_s2,
            test_pred,
            target_names=[
                "Outer",
                "Inner",
                "Both"
            ],
            zero_division=0
        )
    )

stage2_test_df = pd.DataFrame(
    stage2_test_results
)

print("\nFinal Stage 2 Teacher Test Comparison")
display(stage2_test_df)

In [ ]:
# ============================================================
# Stage 2 Test Performance by Physical Bearing
# ============================================================

stage2_test_bearings = ["KA22", "KI18", "KI21", "KB27"]

per_bearing_stage2 = []

for model_name, model in stage2_teacher_models.items():

    test_prob = model.predict(
        X_test_s2,
        verbose=0
    )

    test_pred = np.argmax(
        test_prob,
        axis=1
    )

    start = 0

    for bearing in stage2_test_bearings:

        end = start + 64

        bearing_true = y_test_s2[start:end]
        bearing_pred = test_pred[start:end]

        per_bearing_stage2.append({
            "model": model_name,
            "bearing": bearing,
            "accuracy": accuracy_score(
                bearing_true,
                bearing_pred
            )
        })

        start = end

per_bearing_stage2_df = pd.DataFrame(
    per_bearing_stage2
)

display(
    per_bearing_stage2_df.pivot(
        index="bearing",
        columns="model",
        values="accuracy"
    )
)

In [ ]:
stage2_test_df.to_csv(
    os.path.join(
        results_dir,
        "stage2_teacher_test_results.csv"
    ),
    index=False
)

per_bearing_stage2_df.to_csv(
    os.path.join(
        results_dir,
        "stage2_teacher_per_bearing_test_results.csv"
    ),
    index=False
)

print("Stage 2 test results saved.")

## Stage 2 Hyperparameter Tuning

The initial Stage 2 LSTM and Transformer models are retained as baseline
experiments. Further model development uses only the training and validation
sets. The frozen test set is reserved for final evaluation after tuning.

In [ ]:
# ============================================================
# Stage 2 Tuning Experiment 1: Class Weights
# Validation set only
# ============================================================

class_weight_experiments = {
    "unweighted": {
        0: 1.0,
        1: 1.0,
        2: 1.0
    },
    "mild_both": {
        0: 1.0,
        1: 1.0,
        2: 2.0
    },
    "moderate_both": {
        0: 1.0,
        1: 1.0,
        2: 3.0
    },
    "strong_both": {
        0: 1.0,
        1: 1.0,
        2: 4.0
    },
    "balanced_current": class_weight_dict_s2
}

for name, weights in class_weight_experiments.items():
    print(f"{name}: {weights}")

In [ ]:
# ============================================================
# Stage 2 LSTM - Class Weight Tuning
# ============================================================

lstm_tuning_results = []

for weight_name, weights in class_weight_experiments.items():

    print("\n" + "=" * 60)
    print(f"LSTM | Class-weight setting: {weight_name}")
    print("=" * 60)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_lstm_stage2()

    early_stopping_tune = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=weights,
        callbacks=[early_stopping_tune],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "weight_setting": weight_name,
        "weights": str(weights),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    lstm_tuning_results.append(result)

    print(result)
    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
lstm_tuning_df = pd.DataFrame(
    lstm_tuning_results
)

display(
    lstm_tuning_df[
        [
            "weight_setting",
            "epochs",
            "val_accuracy",
            "val_balanced_accuracy",
            "val_macro_precision",
            "val_macro_recall",
            "val_macro_f1"
        ]
    ]
)

In [ ]:
# ============================================================
# Stage 2 LSTM - Model Capacity Tuning
# Class weights = None
# Only LSTM hidden units are changed.
# ============================================================

def build_lstm_stage2_capacity(lstm_units, dense_units=32):

    model = keras.Sequential([
        layers.Input(shape=(5, 12)),
        layers.LSTM(lstm_units),
        layers.Dropout(0.2),
        layers.Dense(dense_units, activation="relu"),
        layers.Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
lstm_capacity_results = []

lstm_units_list = [32, 64, 128]

for units in lstm_units_list:

    print("\n" + "=" * 60)
    print(f"LSTM | Hidden units = {units}")
    print("=" * 60)

    # Fresh model for every capacity
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_lstm_stage2_capacity(
        lstm_units=units,
        dense_units=32
    )

    early_stopping_capacity = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=None,
        callbacks=[early_stopping_capacity],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "lstm_units": units,
        "parameters": model.count_params(),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    lstm_capacity_results.append(result)

    print(result)

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
lstm_capacity_df = pd.DataFrame(
    lstm_capacity_results
)

display(lstm_capacity_df)

In [ ]:
# ============================================================
# Stage 2 LSTM - Learning Rate Tuning
# Best capacity so far: 32 LSTM units
# Class weights = None
# ============================================================

def build_lstm_stage2_lr(learning_rate):

    model = keras.Sequential([
        layers.Input(shape=(5, 12)),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
lstm_lr_results = []

learning_rates = [0.005, 0.01]

for lr in learning_rates:

    print("\n" + "=" * 60)
    print(f"LSTM | Learning rate = {lr}")
    print("=" * 60)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_lstm_stage2_lr(lr)

    early_stopping_lr = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=None,
        callbacks=[early_stopping_lr],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "learning_rate": lr,
        "parameters": model.count_params(),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    lstm_lr_results.append(result)

    print(result)

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
lstm_lr_df = pd.DataFrame(lstm_lr_results)

display(lstm_lr_df)

In [ ]:
# ============================================================
# Stage 2 LSTM - Dropout Tuning
# Best settings so far:
# LSTM units = 32
# Learning rate = 0.003
# Class weights = None
# ============================================================

def build_lstm_stage2_dropout(dropout_rate):

    model = keras.Sequential([
        layers.Input(shape=(5, 12)),
        layers.LSTM(32),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.003
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
lstm_dropout_results = []

dropout_rates = [0.0, 0.1, 0.3, 0.4]

for dropout in dropout_rates:

    print("\n" + "=" * 60)
    print(f"LSTM | Dropout = {dropout}")
    print("=" * 60)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_lstm_stage2_dropout(dropout)

    early_stopping_dropout = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=None,
        callbacks=[early_stopping_dropout],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "dropout": dropout,
        "parameters": model.count_params(),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    lstm_dropout_results.append(result)

    print(result)

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
lstm_dropout_df = pd.DataFrame(
    lstm_dropout_results
)

display(lstm_dropout_df)

In [ ]:
# ============================================================
# Stage 2 Transformer - Class Weight Tuning
# Validation set only
# ============================================================

transformer_tuning_results = []

for weight_name, weights in class_weight_experiments.items():

    print("\n" + "=" * 60)
    print(f"Transformer | Class-weight setting: {weight_name}")
    print("=" * 60)

    # Fresh Transformer with the same initialization
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_transformer_stage2()

    early_stopping_tune = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=weights,
        callbacks=[early_stopping_tune],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "weight_setting": weight_name,
        "weights": str(weights),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    transformer_tuning_results.append(result)

    print(result)
    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
transformer_tuning_df = pd.DataFrame(
    transformer_tuning_results
)

display(
    transformer_tuning_df[
        [
            "weight_setting",
            "epochs",
            "val_accuracy",
            "val_balanced_accuracy",
            "val_macro_precision",
            "val_macro_recall",
            "val_macro_f1"
        ]
    ]
)

In [ ]:
# ============================================================
# Stage 2 Transformer - Model Capacity Tuning
# Class weights = None
# ============================================================

def build_transformer_stage2_capacity(
    model_dim=32,
    num_heads=2,
    ffn_dim=64
):
    transformer_input = keras.Input(shape=(5, 12))

    x = layers.Dense(model_dim)(transformer_input)

    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=model_dim // num_heads,
        dropout=0.1
    )(x, x)

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(ffn_dim, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(model_dim)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    output = layers.Dense(
        3,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs=transformer_input,
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
transformer_capacity_configs = [
    {
        "name": "small",
        "model_dim": 16,
        "num_heads": 2,
        "ffn_dim": 32
    },
    {
        "name": "medium",
        "model_dim": 32,
        "num_heads": 2,
        "ffn_dim": 64
    },
    {
        "name": "large",
        "model_dim": 64,
        "num_heads": 4,
        "ffn_dim": 128
    }
]

transformer_capacity_results = []

for config in transformer_capacity_configs:

    print("\n" + "=" * 60)
    print(
        f"Transformer | {config['name']} | "
        f"dim={config['model_dim']} | "
        f"heads={config['num_heads']} | "
        f"FFN={config['ffn_dim']}"
    )
    print("=" * 60)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_transformer_stage2_capacity(
        model_dim=config["model_dim"],
        num_heads=config["num_heads"],
        ffn_dim=config["ffn_dim"]
    )

    early_stopping_capacity = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=None,
        callbacks=[early_stopping_capacity],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "configuration": config["name"],
        "model_dim": config["model_dim"],
        "num_heads": config["num_heads"],
        "ffn_dim": config["ffn_dim"],
        "parameters": model.count_params(),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    transformer_capacity_results.append(result)

    print(result)

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
transformer_capacity_df = pd.DataFrame(
    transformer_capacity_results
)

display(transformer_capacity_df)

In [ ]:
# ============================================================
# Stage 2 Transformer - Learning Rate Tuning
# Best capacity so far:
# model_dim = 32, heads = 2, FFN = 64
# Class weights = None
# ============================================================

def build_transformer_stage2_lr(learning_rate):

    transformer_input = keras.Input(shape=(5, 12))

    x = layers.Dense(32)(transformer_input)

    attention_output = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.1
    )(x, x)

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(64, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(32)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    output = layers.Dense(
        3,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs=transformer_input,
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
transformer_lr_results = []

learning_rates_transformer = [0.005, 0.01]

for lr in learning_rates_transformer:

    print("\n" + "=" * 60)
    print(f"Transformer | Learning rate = {lr}")
    print("=" * 60)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_transformer_stage2_lr(lr)

    early_stopping_lr = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=None,
        callbacks=[early_stopping_lr],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "learning_rate": lr,
        "parameters": model.count_params(),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(y_val_s2, val_pred),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2, val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    transformer_lr_results.append(result)

    print(result)

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
transformer_lr_df = pd.DataFrame(
    transformer_lr_results
)

display(transformer_lr_df)

In [ ]:
# ============================================================
# Stage 2 Transformer - Dropout Tuning
#
# Fixed:
# model_dim = 32
# num_heads = 2
# ffn_dim = 64
# learning_rate = 0.003
# class weights = None
# batch size = 32
#
# Only the final classifier dropout is changed.
# ============================================================

def build_transformer_stage2_dropout(dropout_rate):

    transformer_input = keras.Input(shape=(5, 12))

    x = layers.Dense(32)(transformer_input)

    attention_output = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.1
    )(x, x)

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(64, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(32)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)

    # Only this dropout is being tuned
    x = layers.Dropout(dropout_rate)(x)

    output = layers.Dense(
        3,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs=transformer_input,
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.003
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
transformer_dropout_results = []

dropout_rates_transformer = [0.0, 0.1, 0.3, 0.4]

for dropout in dropout_rates_transformer:

    print("\n" + "=" * 60)
    print(f"Transformer | Final dropout = {dropout}")
    print("=" * 60)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(10)

    model = build_transformer_stage2_dropout(dropout)

    early_stopping_dropout = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_s2,
        y_train_s2,
        validation_data=(X_val_s2, y_val_s2),
        epochs=50,
        batch_size=32,
        class_weight=None,
        callbacks=[early_stopping_dropout],
        verbose=0
    )

    val_prob = model.predict(
        X_val_s2,
        verbose=0
    )

    val_pred = np.argmax(
        val_prob,
        axis=1
    )

    result = {
        "dropout": dropout,
        "parameters": model.count_params(),
        "epochs": len(history.history["loss"]),
        "val_accuracy": accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_balanced_accuracy": balanced_accuracy_score(
            y_val_s2,
            val_pred
        ),
        "val_macro_precision": precision_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_recall": recall_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1": f1_score(
            y_val_s2,
            val_pred,
            average="macro",
            zero_division=0
        )
    }

    transformer_dropout_results.append(result)

    print(result)

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_val_s2,
            val_pred
        )
    )

In [ ]:
transformer_dropout_df = pd.DataFrame(
    transformer_dropout_results
)

display(transformer_dropout_df)

In [ ]:
# ============================================================
# Final Tuned Stage 2 LSTM
#
# Top12 features
# LSTM units = 32
# Dropout = 0.1
# Learning rate = 0.003
# Batch size = 32
# Class weights = None
# Seed = 10
# ============================================================

def build_final_lstm_stage2():

    model = keras.Sequential([
        layers.Input(shape=(5, 12)),
        layers.LSTM(32),
        layers.Dropout(0.1),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.003
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
# Train final tuned Stage 2 LSTM

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(10)

final_lstm_stage2 = build_final_lstm_stage2()

early_stopping_final_lstm = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

final_lstm_stage2_history = final_lstm_stage2.fit(
    X_train_s2,
    y_train_s2,
    validation_data=(X_val_s2, y_val_s2),
    epochs=50,
    batch_size=32,
    class_weight=None,
    callbacks=[early_stopping_final_lstm],
    verbose=1
)

print("\nFinal tuned Stage 2 LSTM training complete.")
print(
    "Epochs actually run:",
    len(final_lstm_stage2_history.history["loss"])
)
print(
    "Parameters:",
    final_lstm_stage2.count_params()
)

In [ ]:
# Save final tuned Stage 2 LSTM

final_lstm_stage2_path = os.path.join(
    results_dir,
    "stage2_lstm_teacher_tuned_final.keras"
)

final_lstm_stage2.save(final_lstm_stage2_path)

print("Saved:")
print(final_lstm_stage2_path)

In [ ]:
# ============================================================
# Final Tuned Stage 2 LSTM - Validation Evaluation
# ============================================================

val_prob_final_lstm = final_lstm_stage2.predict(
    X_val_s2,
    verbose=0
)

val_pred_final_lstm = np.argmax(
    val_prob_final_lstm,
    axis=1
)

print("Final Tuned Stage 2 LSTM - Validation Results")
print("------------------------------------------------")
print(
    "Accuracy:          ",
    accuracy_score(y_val_s2, val_pred_final_lstm)
)
print(
    "Balanced Accuracy: ",
    balanced_accuracy_score(
        y_val_s2,
        val_pred_final_lstm
    )
)
print(
    "Macro Precision:   ",
    precision_score(
        y_val_s2,
        val_pred_final_lstm,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro Recall:      ",
    recall_score(
        y_val_s2,
        val_pred_final_lstm,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro F1:          ",
    f1_score(
        y_val_s2,
        val_pred_final_lstm,
        average="macro",
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_s2,
        val_pred_final_lstm
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_val_s2,
        val_pred_final_lstm,
        target_names=["Outer", "Inner", "Both"],
        zero_division=0
    )
)

In [ ]:
# ============================================================
# Final Tuned Stage 2 Transformer
# ============================================================

def build_final_transformer_stage2():

    transformer_input = keras.Input(shape=(5, 12))

    x = layers.Dense(32)(transformer_input)

    attention_output = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.1
    )(x, x)

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(64, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(32)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    output = layers.Dense(
        3,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs=transformer_input,
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.003
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
# Train final tuned Stage 2 Transformer

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(10)

final_transformer_stage2 = build_final_transformer_stage2()

early_stopping_final_transformer = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

final_transformer_stage2_history = final_transformer_stage2.fit(
    X_train_s2,
    y_train_s2,
    validation_data=(X_val_s2, y_val_s2),
    epochs=50,
    batch_size=32,
    class_weight=None,
    callbacks=[early_stopping_final_transformer],
    verbose=1
)

print("\nFinal tuned Stage 2 Transformer training complete.")
print(
    "Epochs actually run:",
    len(final_transformer_stage2_history.history["loss"])
)
print(
    "Parameters:",
    final_transformer_stage2.count_params()
)

In [ ]:
# Save final tuned Stage 2 Transformer

final_transformer_stage2_path = os.path.join(
    results_dir,
    "stage2_transformer_teacher_tuned_final.keras"
)

final_transformer_stage2.save(
    final_transformer_stage2_path
)

print("Saved:")
print(final_transformer_stage2_path)

In [ ]:
# ============================================================
# Final Tuned Stage 2 Transformer - Validation Evaluation
# ============================================================

val_prob_final_transformer = final_transformer_stage2.predict(
    X_val_s2,
    verbose=0
)

val_pred_final_transformer = np.argmax(
    val_prob_final_transformer,
    axis=1
)

print("Final Tuned Stage 2 Transformer - Validation Results")
print("-----------------------------------------------------")
print(
    "Accuracy:          ",
    accuracy_score(
        y_val_s2,
        val_pred_final_transformer
    )
)
print(
    "Balanced Accuracy: ",
    balanced_accuracy_score(
        y_val_s2,
        val_pred_final_transformer
    )
)
print(
    "Macro Precision:   ",
    precision_score(
        y_val_s2,
        val_pred_final_transformer,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro Recall:      ",
    recall_score(
        y_val_s2,
        val_pred_final_transformer,
        average="macro",
        zero_division=0
    )
)
print(
    "Macro F1:          ",
    f1_score(
        y_val_s2,
        val_pred_final_transformer,
        average="macro",
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_s2,
        val_pred_final_transformer
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_val_s2,
        val_pred_final_transformer,
        target_names=["Outer", "Inner", "Both"],
        zero_division=0
    )
)

print(
    "\nSaved model exists:",
    os.path.exists(final_transformer_stage2_path)
)

In [ ]:
# ============================================================
# FINAL Stage 2 Test Evaluation
# Tuned models — test set used only now
# ============================================================

final_stage2_models = {
    "LSTM_Tuned": keras.models.load_model(
        final_lstm_stage2_path
    ),
    "Transformer_Tuned": keras.models.load_model(
        final_transformer_stage2_path
    )
}

final_stage2_test_results = []

for model_name, model in final_stage2_models.items():

    test_prob = model.predict(
        X_test_s2,
        verbose=0
    )

    test_pred = np.argmax(
        test_prob,
        axis=1
    )

    result = {
        "model": model_name,
        "parameters": model.count_params(),
        "test_accuracy": accuracy_score(
            y_test_s2,
            test_pred
        ),
        "test_balanced_accuracy": balanced_accuracy_score(
            y_test_s2,
            test_pred
        ),
        "test_macro_precision": precision_score(
            y_test_s2,
            test_pred,
            average="macro",
            zero_division=0
        ),
        "test_macro_recall": recall_score(
            y_test_s2,
            test_pred,
            average="macro",
            zero_division=0
        ),
        "test_macro_f1": f1_score(
            y_test_s2,
            test_pred,
            average="macro",
            zero_division=0
        )
    }

    final_stage2_test_results.append(result)

    print("\n" + "=" * 60)
    print(f"{model_name} - FINAL Stage 2 TEST")
    print("=" * 60)

    print("Accuracy:          ", result["test_accuracy"])
    print("Balanced Accuracy: ", result["test_balanced_accuracy"])
    print("Macro Precision:   ", result["test_macro_precision"])
    print("Macro Recall:      ", result["test_macro_recall"])
    print("Macro F1:           ", result["test_macro_f1"])

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_test_s2,
            test_pred
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test_s2,
            test_pred,
            target_names=["Outer", "Inner", "Both"],
            zero_division=0
        )
    )

final_stage2_test_df = pd.DataFrame(
    final_stage2_test_results
)

print("\nFinal Stage 2 Test Comparison")
display(final_stage2_test_df)

In [ ]:
final_stage2_test_df.to_csv(
    os.path.join(
        results_dir,
        "stage2_tuned_teacher_test_results.csv"
    ),
    index=False
)

print("Final Stage 2 test results saved.")

## Stage 2 Multi-Label Diagnosis

The previous Stage 2 experiment treated Outer, Inner, and Both as three
mutually exclusive classes.

The new formulation predicts the presence of Outer-ring and Inner-ring
damage independently:

- Output 0: Outer damage present
- Output 1: Inner damage present

Therefore:
- Outer  = [1, 0]
- Inner  = [0, 1]
- Both   = [1, 1]

The existing bearing-level train, validation, and test splits are retained.

In [ ]:
# ============================================================
# Stage 2 Multi-Label Targets
#
# Existing labels:
# 0 = Outer
# 1 = Inner
# 2 = Both
#
# New labels:
# [1, 0] = Outer
# [0, 1] = Inner
# [1, 1] = Both
# ============================================================

def create_multilabel_targets(y):
    y = np.asarray(y)

    outer_present = np.isin(y, [0, 2]).astype(np.float32)
    inner_present = np.isin(y, [1, 2]).astype(np.float32)

    return np.column_stack([
        outer_present,
        inner_present
    ])


y_train_multi = create_multilabel_targets(y_train_s2)
y_val_multi = create_multilabel_targets(y_val_s2)
y_test_multi = create_multilabel_targets(y_test_s2)

print("Train shape:", y_train_multi.shape)
print("Validation shape:", y_val_multi.shape)
print("Test shape:", y_test_multi.shape)

print("\nTraining label counts:")
print("Outer present :", int(y_train_multi[:, 0].sum()))
print("Outer absent  :", int(len(y_train_multi) - y_train_multi[:, 0].sum()))
print("Inner present :", int(y_train_multi[:, 1].sum()))
print("Inner absent  :", int(len(y_train_multi) - y_train_multi[:, 1].sum()))

print("\nValidation label counts:")
print("Outer present :", int(y_val_multi[:, 0].sum()))
print("Outer absent  :", int(len(y_val_multi) - y_val_multi[:, 0].sum()))
print("Inner present :", int(y_val_multi[:, 1].sum()))
print("Inner absent  :", int(len(y_val_multi) - y_val_multi[:, 1].sum()))

print("\nTest label counts:")
print("Outer present :", int(y_test_multi[:, 0].sum()))
print("Outer absent  :", int(len(y_test_multi) - y_test_multi[:, 0].sum()))
print("Inner present :", int(y_test_multi[:, 1].sum()))
print("Inner absent  :", int(len(y_test_multi) - y_test_multi[:, 1].sum()))

print("\nFirst 10 original -> multi-label mappings:")
for original, multi in zip(y_train_s2[:10], y_train_multi[:10]):
    print(f"{int(original)} -> {multi.astype(int)}")

In [ ]:
# ============================================================
# Stage 2 Multi-Label LSTM
#
# Output 0 = Outer damage present
# Output 1 = Inner damage present
# ============================================================

def build_multilabel_lstm():

    model = keras.Sequential([
        layers.Input(shape=(5, 12)),
        layers.LSTM(32),
        layers.Dropout(0.1),
        layers.Dense(32, activation="relu"),
        layers.Dense(2, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.003
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


multilabel_lstm = build_multilabel_lstm()

multilabel_lstm.summary()

In [ ]:
# ============================================================
# Train Stage 2 Multi-Label LSTM
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(10)

multilabel_lstm = build_multilabel_lstm()

early_stopping_multi_lstm = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

multilabel_lstm_history = multilabel_lstm.fit(
    X_train_s2,
    y_train_multi,
    validation_data=(X_val_s2, y_val_multi),
    epochs=50,
    batch_size=32,
    class_weight=None,
    callbacks=[early_stopping_multi_lstm],
    verbose=1
)

print("\nMulti-label LSTM training complete.")
print(
    "Epochs actually run:",
    len(multilabel_lstm_history.history["loss"])
)
print(
    "Parameters:",
    multilabel_lstm.count_params()
)

In [ ]:
# ============================================================
# Stage 2 Multi-Label LSTM - Validation Evaluation
# ============================================================

val_prob_multi = multilabel_lstm.predict(
    X_val_s2,
    verbose=0
)

# Convert probabilities to binary predictions
val_pred_multi = (val_prob_multi >= 0.5).astype(int)

# ------------------------------------------------------------
# Per-output evaluation
# ------------------------------------------------------------

print("OUTER DAMAGE DETECTOR")
print("---------------------")
print(
    "Accuracy:",
    accuracy_score(
        y_val_multi[:, 0],
        val_pred_multi[:, 0]
    )
)
print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_val_multi[:, 0],
        val_pred_multi[:, 0]
    )
)
print(
    "F1:",
    f1_score(
        y_val_multi[:, 0],
        val_pred_multi[:, 0],
        zero_division=0
    )
)
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_multi[:, 0],
        val_pred_multi[:, 0]
    )
)

print("\n\nINNER DAMAGE DETECTOR")
print("---------------------")
print(
    "Accuracy:",
    accuracy_score(
        y_val_multi[:, 1],
        val_pred_multi[:, 1]
    )
)
print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_val_multi[:, 1],
        val_pred_multi[:, 1]
    )
)
print(
    "F1:",
    f1_score(
        y_val_multi[:, 1],
        val_pred_multi[:, 1],
        zero_division=0
    )
)
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_multi[:, 1],
        val_pred_multi[:, 1]
    )
)

# ------------------------------------------------------------
# Reconstruct final diagnosis
# ------------------------------------------------------------

def multilabel_to_fault(outer, inner):
    if outer == 1 and inner == 0:
        return 0   # Outer
    elif outer == 0 and inner == 1:
        return 1   # Inner
    elif outer == 1 and inner == 1:
        return 2   # Both
    else:
        return -1  # Invalid for damaged-only Stage 2

val_fault_true = np.array([
    multilabel_to_fault(int(outer), int(inner))
    for outer, inner in y_val_multi
])

val_fault_pred = np.array([
    multilabel_to_fault(int(outer), int(inner))
    for outer, inner in val_pred_multi
])

# ------------------------------------------------------------
# Exact diagnosis evaluation
# ------------------------------------------------------------

print("\n\nRECONSTRUCTED FAULT DIAGNOSIS")
print("-----------------------------")
print(
    "Exact diagnosis accuracy:",
    accuracy_score(
        val_fault_true,
        val_fault_pred
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        val_fault_true,
        val_fault_pred
    )
)

print(
    "Macro F1:",
    f1_score(
        val_fault_true,
        val_fault_pred,
        average="macro",
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        val_fault_true,
        val_fault_pred,
        labels=[0, 1, 2]
    )
)

print("\nClassification Report:")
print(
    classification_report(
        val_fault_true,
        val_fault_pred,
        labels=[0, 1, 2],
        target_names=["Outer", "Inner", "Both"],
        zero_division=0
    )
)

# ------------------------------------------------------------
# Exact-match ratio
# Both outputs must be correct
# ------------------------------------------------------------

exact_match = np.all(
    y_val_multi == val_pred_multi,
    axis=1
)

print(
    "\nMulti-label exact-match accuracy:",
    exact_match.mean()
)

In [ ]:
# ============================================================
# Stage 2 Multi-Label LSTM - Clean Threshold Tuning
# Validation set only
#
# We evaluate the two binary outputs directly:
#   Output 0 = Outer damage
#   Output 1 = Inner damage
# ============================================================

thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for outer_threshold in thresholds:
    for inner_threshold in thresholds:

        # Binary predictions for each detector
        outer_pred = (
            val_prob_multi[:, 0] >= outer_threshold
        ).astype(int)

        inner_pred = (
            val_prob_multi[:, 1] >= inner_threshold
        ).astype(int)

        # Exact multi-label match
        combined_pred = np.column_stack([
            outer_pred,
            inner_pred
        ])

        exact_match = np.mean(
            np.all(
                y_val_multi == combined_pred,
                axis=1
            )
        )

        # Binary F1 for each detector
        outer_f1 = f1_score(
            y_val_multi[:, 0],
            outer_pred,
            zero_division=0
        )

        inner_f1 = f1_score(
            y_val_multi[:, 1],
            inner_pred,
            zero_division=0
        )

        macro_binary_f1 = (
            outer_f1 + inner_f1
        ) / 2

        # Binary balanced accuracy for each detector
        outer_balanced_acc = balanced_accuracy_score(
            y_val_multi[:, 0],
            outer_pred
        )

        inner_balanced_acc = balanced_accuracy_score(
            y_val_multi[:, 1],
            inner_pred
        )

        mean_balanced_acc = (
            outer_balanced_acc + inner_balanced_acc
        ) / 2

        # Invalid Stage-2 diagnosis: [0, 0]
        invalid_predictions = np.sum(
            np.all(combined_pred == [0, 0], axis=1)
        )

        threshold_results.append({
            "outer_threshold": outer_threshold,
            "inner_threshold": inner_threshold,
            "exact_match_accuracy": exact_match,
            "outer_f1": outer_f1,
            "inner_f1": inner_f1,
            "macro_binary_f1": macro_binary_f1,
            "outer_balanced_accuracy": outer_balanced_acc,
            "inner_balanced_accuracy": inner_balanced_acc,
            "mean_balanced_accuracy": mean_balanced_acc,
            "invalid_predictions": int(invalid_predictions)
        })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values(
        ["macro_binary_f1", "exact_match_accuracy"],
        ascending=False
    ).head(10)
)

In [ ]:
best_threshold = threshold_df.sort_values(
    ["macro_binary_f1", "exact_match_accuracy"],
    ascending=False
).iloc[0]

print("Best threshold configuration:")
print(best_threshold)

In [ ]:
# ============================================================
# Select threshold using FINAL DIAGNOSIS accuracy
# ============================================================

display(
    threshold_df.sort_values(
        ["exact_match_accuracy", "macro_binary_f1"],
        ascending=False
    ).head(10)
)

In [ ]:
best_diagnosis_threshold = threshold_df.sort_values(
    ["exact_match_accuracy", "macro_binary_f1"],
    ascending=False
).iloc[0]

print("Best threshold for final diagnosis:")
print(best_diagnosis_threshold)

In [ ]:
# ============================================================
# Stage 2 Multi-Label Transformer
#
# Output 0 = Outer damage present
# Output 1 = Inner damage present
#
# Configuration:
# model_dim = 32
# heads = 2
# FFN = 64
# learning rate = 0.003
# final dropout = 0.3
# batch size = 32
# seed = 10
# ============================================================

def build_multilabel_transformer():

    transformer_input = keras.Input(shape=(5, 12))

    x = layers.Dense(32)(transformer_input)

    attention_output = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.1
    )(x, x)

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(64, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(32)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    output = layers.Dense(
        2,
        activation="sigmoid"
    )(x)

    model = keras.Model(
        inputs=transformer_input,
        outputs=output
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.003
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


multilabel_transformer = build_multilabel_transformer()

multilabel_transformer.summary()

In [ ]:
# ============================================================
# Train Stage 2 Multi-Label Transformer
#
# Output 0 = Outer damage present
# Output 1 = Inner damage present
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(10)

multilabel_transformer = build_multilabel_transformer()

early_stopping_multi_transformer = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

multilabel_transformer_history = multilabel_transformer.fit(
    X_train_s2,
    y_train_multi,
    validation_data=(X_val_s2, y_val_multi),
    epochs=50,
    batch_size=32,
    class_weight=None,
    callbacks=[early_stopping_multi_transformer],
    verbose=1
)

print("\nMulti-label Transformer training complete.")
print(
    "Epochs actually run:",
    len(multilabel_transformer_history.history["loss"])
)
print(
    "Parameters:",
    multilabel_transformer.count_params()
)

In [ ]:
# ============================================================
# Save Stage 2 Multi-Label Transformer
# ============================================================

multilabel_transformer_path = os.path.join(
    results_dir,
    "stage2_multilabel_transformer_teacher.keras"
)

multilabel_transformer.save(
    multilabel_transformer_path
)

print("Saved:")
print(multilabel_transformer_path)
print("Parameters:", multilabel_transformer.count_params())

In [ ]:
# ============================================================
# Stage 2 Multi-Label Transformer - Validation Evaluation
# ============================================================

val_prob_multi_transformer = multilabel_transformer.predict(
    X_val_s2,
    verbose=0
)

val_pred_multi_transformer = (
    val_prob_multi_transformer >= 0.5
).astype(int)

# ------------------------------------------------------------
# Outer detector
# ------------------------------------------------------------

print("OUTER DAMAGE DETECTOR")
print("---------------------")

print(
    "Accuracy:",
    accuracy_score(
        y_val_multi[:, 0],
        val_pred_multi_transformer[:, 0]
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_val_multi[:, 0],
        val_pred_multi_transformer[:, 0]
    )
)

print(
    "F1:",
    f1_score(
        y_val_multi[:, 0],
        val_pred_multi_transformer[:, 0],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_multi[:, 0],
        val_pred_multi_transformer[:, 0]
    )
)

# ------------------------------------------------------------
# Inner detector
# ------------------------------------------------------------

print("\n\nINNER DAMAGE DETECTOR")
print("---------------------")

print(
    "Accuracy:",
    accuracy_score(
        y_val_multi[:, 1],
        val_pred_multi_transformer[:, 1]
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_val_multi[:, 1],
        val_pred_multi_transformer[:, 1]
    )
)

print(
    "F1:",
    f1_score(
        y_val_multi[:, 1],
        val_pred_multi_transformer[:, 1],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val_multi[:, 1],
        val_pred_multi_transformer[:, 1]
    )
)

# ------------------------------------------------------------
# Reconstruct Outer / Inner / Both
# ------------------------------------------------------------

val_fault_true_multi = np.array([
    multilabel_to_fault(int(o), int(i))
    for o, i in y_val_multi
])

val_fault_pred_multi_transformer = np.array([
    multilabel_to_fault(int(o), int(i))
    for o, i in val_pred_multi_transformer
])

valid = val_fault_pred_multi_transformer != -1

print("\n\nRECONSTRUCTED FAULT DIAGNOSIS")
print("-----------------------------")

print(
    "Exact diagnosis accuracy:",
    np.mean(
        val_fault_true_multi ==
        val_fault_pred_multi_transformer
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        val_fault_true_multi[valid],
        val_fault_pred_multi_transformer[valid]
    )
)

print(
    "Macro F1:",
    f1_score(
        val_fault_true_multi[valid],
        val_fault_pred_multi_transformer[valid],
        average="macro",
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        val_fault_true_multi[valid],
        val_fault_pred_multi_transformer[valid],
        labels=[0, 1, 2]
    )
)

print("\nClassification Report:")
print(
    classification_report(
        val_fault_true_multi[valid],
        val_fault_pred_multi_transformer[valid],
        labels=[0, 1, 2],
        target_names=["Outer", "Inner", "Both"],
        zero_division=0
    )
)

print(
    "\nInvalid [0,0] predictions:",
    np.sum(~valid)
)

In [ ]:
# ============================================================
# Stage 2 Multi-Label Transformer - Threshold Tuning
# Validation set only
# ============================================================

thresholds = np.arange(0.20, 0.81, 0.05)

transformer_threshold_results = []

for outer_threshold in thresholds:
    for inner_threshold in thresholds:

        outer_pred = (
            val_prob_multi_transformer[:, 0] >= outer_threshold
        ).astype(int)

        inner_pred = (
            val_prob_multi_transformer[:, 1] >= inner_threshold
        ).astype(int)

        combined_pred = np.column_stack([
            outer_pred,
            inner_pred
        ])

        # Convert to Outer / Inner / Both
        fault_pred = np.array([
            multilabel_to_fault(int(o), int(i))
            for o, i in combined_pred
        ])

        fault_true = np.array([
            multilabel_to_fault(int(o), int(i))
            for o, i in y_val_multi
        ])

        # Invalid [0,0] predictions
        invalid = fault_pred == -1

        # Exact diagnosis accuracy includes invalid predictions
        exact_accuracy = np.mean(
            fault_pred == fault_true
        )

        # Binary metrics
        outer_f1 = f1_score(
            y_val_multi[:, 0],
            outer_pred,
            zero_division=0
        )

        inner_f1 = f1_score(
            y_val_multi[:, 1],
            inner_pred,
            zero_division=0
        )

        outer_balanced = balanced_accuracy_score(
            y_val_multi[:, 0],
            outer_pred
        )

        inner_balanced = balanced_accuracy_score(
            y_val_multi[:, 1],
            inner_pred
        )

        transformer_threshold_results.append({
            "outer_threshold": outer_threshold,
            "inner_threshold": inner_threshold,
            "exact_match_accuracy": exact_accuracy,
            "outer_f1": outer_f1,
            "inner_f1": inner_f1,
            "macro_binary_f1": (
                outer_f1 + inner_f1
            ) / 2,
            "outer_balanced_accuracy": outer_balanced,
            "inner_balanced_accuracy": inner_balanced,
            "mean_balanced_accuracy": (
                outer_balanced + inner_balanced
            ) / 2,
            "invalid_predictions": int(invalid.sum())
        })

transformer_threshold_df = pd.DataFrame(
    transformer_threshold_results
)

display(
    transformer_threshold_df.sort_values(
        ["exact_match_accuracy", "macro_binary_f1"],
        ascending=False
    ).head(10)
)

In [ ]:
best_transformer_threshold = transformer_threshold_df.sort_values(
    ["exact_match_accuracy", "macro_binary_f1"],
    ascending=False
).iloc[0]

print("Best Transformer threshold configuration:")
print(best_transformer_threshold)

In [ ]:
# ============================================================
# Freeze final Stage 2 Multi-Label configurations
# ============================================================

FINAL_LSTM_OUTER_THRESHOLD = 0.45
FINAL_LSTM_INNER_THRESHOLD = 0.55

FINAL_TRANSFORMER_OUTER_THRESHOLD = 0.70
FINAL_TRANSFORMER_INNER_THRESHOLD = 0.20

print("Final Multi-Label LSTM thresholds:")
print("Outer:", FINAL_LSTM_OUTER_THRESHOLD)
print("Inner:", FINAL_LSTM_INNER_THRESHOLD)

print("\nFinal Multi-Label Transformer thresholds:")
print("Outer:", FINAL_TRANSFORMER_OUTER_THRESHOLD)
print("Inner:", FINAL_TRANSFORMER_INNER_THRESHOLD)

In [ ]:
# ============================================================
# FINAL Stage 2 Multi-Label TEST Evaluation
# Test set used only after validation tuning is frozen.
# ============================================================

final_multilabel_models = {
    "LSTM": keras.models.load_model(
        multilabel_lstm_path
    ) if "multilabel_lstm_path" in globals()
    else multilabel_lstm,

    "Transformer": keras.models.load_model(
        multilabel_transformer_path
    )
}

threshold_config = {
    "LSTM": (
        FINAL_LSTM_OUTER_THRESHOLD,
        FINAL_LSTM_INNER_THRESHOLD
    ),
    "Transformer": (
        FINAL_TRANSFORMER_OUTER_THRESHOLD,
        FINAL_TRANSFORMER_INNER_THRESHOLD
    )
}

final_multilabel_test_results = []

for model_name, model in final_multilabel_models.items():

    test_prob = model.predict(
        X_test_s2,
        verbose=0
    )

    outer_threshold, inner_threshold = threshold_config[
        model_name
    ]

    test_outer = (
        test_prob[:, 0] >= outer_threshold
    ).astype(int)

    test_inner = (
        test_prob[:, 1] >= inner_threshold
    ).astype(int)

    test_multi_pred = np.column_stack([
        test_outer,
        test_inner
    ])

    test_fault_true = np.array([
        multilabel_to_fault(int(o), int(i))
        for o, i in y_test_multi
    ])

    test_fault_pred = np.array([
        multilabel_to_fault(int(o), int(i))
        for o, i in test_multi_pred
    ])

    invalid = test_fault_pred == -1

    result = {
        "model": model_name,
        "parameters": model.count_params(),
        "outer_threshold": outer_threshold,
        "inner_threshold": inner_threshold,
        "exact_diagnosis_accuracy": np.mean(
            test_fault_true == test_fault_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            test_fault_true,
            test_fault_pred
        ),
        "macro_f1": f1_score(
            test_fault_true,
            test_fault_pred,
            average="macro",
            zero_division=0
        ),
        "invalid_predictions": int(invalid.sum())
    }

    final_multilabel_test_results.append(result)

    print("\n" + "=" * 60)
    print(f"{model_name} - FINAL MULTI-LABEL STAGE 2 TEST")
    print("=" * 60)

    print("Outer threshold:", outer_threshold)
    print("Inner threshold:", inner_threshold)

    print(
        "Exact diagnosis accuracy:",
        result["exact_diagnosis_accuracy"]
    )
    print(
        "Balanced accuracy:",
        result["balanced_accuracy"]
    )
    print(
        "Macro F1:",
        result["macro_f1"]
    )
    print(
        "Invalid predictions:",
        result["invalid_predictions"]
    )

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            test_fault_true,
            test_fault_pred,
            labels=[0, 1, 2]
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            test_fault_true,
            test_fault_pred,
            labels=[0, 1, 2],
            target_names=["Outer", "Inner", "Both"],
            zero_division=0
        )
    )

final_multilabel_test_df = pd.DataFrame(
    final_multilabel_test_results
)

print("\nFinal Multi-Label Stage 2 Test Comparison")
display(final_multilabel_test_df)